In [42]:
import json
import os
import sys
import cv2

from pathlib import Path

In [83]:
# 1. Define paths based on your tree structure
json_path = ROOT_DIR / 'data/TAO_Amodal_Dataset/ASPIRe_labels/train.json'
frames_dir = ROOT_DIR / 'data/TAO_Amodal_Dataset/frames'
output_dir = ROOT_DIR / 'data/TAO_Amodal_Dataset/exploration_output'

print("Lendo JSON...")
with open(json_path, 'r') as f:
    dataset = json.load(f)

Lendo JSON...


In [84]:
print("\nEstrutura principal do arquivo:")
for key, value in dataset.items():
    if isinstance(value, list):
        if len(value) > 0:
            # Pega apenas um pedacinho do primeiro item para não poluir a tela
            sample = str(value[0])[:100] + "..." if len(str(value[0])) > 100 else value[0]
            print(f" - '{key}': Lista com {len(value)} itens. (Exemplo: {sample})")
        else:
            print(f" - '{key}': Lista vazia")
    else:
        print(f" - '{key}': {type(value)}")


Estrutura principal do arquivo:
 - 'data': Lista com 17209 itens. (Exemplo: {'file_name': 'train/YFCC100M/v_f69ebe5b731d3e87c1a3992ee39c3b7e/frame0391.jpg', 'pan_seg_file_name'...)
 - 'videos': Lista com 500 itens. (Exemplo: {'id': 0, 'width': 640, 'height': 480, 'neg_category_ids': [342, 57, 651, 357, 738], 'not_exhaustive...)
 - 'thing_classes': Lista com 1225 itens. (Exemplo: acorn)
 - 'stuff_classes': Lista com 5 itens. (Exemplo: banner)
 - 'predicate_apparances': Lista com 722 itens. (Exemplo: abduction)
 - 'predicate_situations': Lista com 2902 itens. (Exemplo: abandoned)
 - 'predicate_positions': Lista com 130 itens. (Exemplo: above)
 - 'predicate_interactions': Lista com 565 itens. (Exemplo: accepting)
 - 'predicate_relations': Lista com 230 itens. (Exemplo: Muslim women)


In [47]:
# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

In [86]:
# # Vamos pegar o primeiro vídeo e ver o que tem dentro
# print("Conteúdo de um item em 'videos':")
# print(json.dumps(dataset['videos'][0], indent=4))

In [79]:
# 1. Carregue as classes
thing_classes = dataset['thing_classes']

# 2. Em vez de enumerate, crie um mapa que lida com o offset do LVIS
# No LVIS, o ID 1 geralmente é a primeira classe ('acorn').
# Então o ID 'i' corresponde ao índice 'i-1'.
category_map = {i + 1: name for i, name in enumerate(thing_classes)}

In [80]:
print(f"ID 805 é: {category_map.get(805)}")
print(f"ID 806 é: {category_map.get(806)}")

ID 805 é: pew_(church_bench)
ID 806 é: phonebook


In [87]:
# 2. Filtrar PRIMEIRO o vídeo para podermos analisar as bounding boxes dele
target_video = "D8Vhxbho1fY_scene_8_128633-130030"
target_data = [item for item in dataset['data'] if target_video in item['file_name']]
print(f"Found {len(target_data)} annotated frames for {target_video}.")

print("\nExtracting text mappings and fixing offset...")

Found 29 annotated frames for D8Vhxbho1fY_scene_8_128633-130030.

Extracting text mappings and fixing offset...


In [88]:
print("Extracting text mappings...")

# --- NEW: Extract Object Categories ---
category_map = {}
if 'thing_classes' in dataset:
    for idx, name in enumerate(dataset['thing_classes']):
        category_map[idx] = name

# --- NEW: Extract Action Categories ---
action_map = {}
if 'predicate_interactions' in dataset:
    for idx, name in enumerate(dataset['predicate_interactions']):
        action_map[idx] = name

Extracting text mappings...


In [64]:
action_map[0]

'accepting'

In [63]:
category_map[805]

'phonebook'

In [65]:
# 2. Filter only the target video
target_video = "D8Vhxbho1fY_scene_8_128633-130030"
# In the JSON, file_name looks like: "train/AVA/D8Vhxbho1fY_scene_8_.../frame0201.jpg"
target_data = [item for item in dataset['data'] if target_video in item['file_name']]

In [66]:
# target_data

In [67]:
print(f"Found {len(target_data)} annotated frames for {target_video}.")

Found 29 annotated frames for D8Vhxbho1fY_scene_8_128633-130030.


In [ ]:
for item in target_data:
    img_path = os.path.join(frames_dir, item['file_name'])
    
    if not os.path.exists(img_path):
        print(f"Image not found: {img_path}")
        continue

    img = cv2.imread(img_path)
    bbox_centers = {}

    # 3. Draw Bounding Boxes with TEXT
    for idx, ann in enumerate(item['annotations']):
        x, y, w, h = map(int, ann['bbox'])
        cat_id = ann['category_id']
        
        # Look up the name, fallback to the number if not found
        cat_name = category_map.get(cat_id, f"Cat:{cat_id}")
        
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(img, f"Obj:{idx} ({cat_name})", (x, y - 5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        bbox_centers[idx] = (x + w // 2, y + h // 2)

    # 4. Draw Interactions with TEXT
    if 'interactions' in item:
        for inter in item['interactions']:
            sub_idx, obj_idx, action_id = inter
            
            if sub_idx in bbox_centers and obj_idx in bbox_centers:
                pt1 = bbox_centers[sub_idx]
                pt2 = bbox_centers[obj_idx]
                
                # Look up the action name, fallback to the number if not found
                act_name = action_map.get(action_id, f"Act:{action_id}")
                
                cv2.line(img, pt1, pt2, (0, 0, 255), 2)
                
                mid_x = (pt1[0] + pt2[0]) // 2
                mid_y = (pt1[1] + pt2[1]) // 2
                
                # We use a white background for text so it's easier to read over the red line
                text_size, _ = cv2.getTextSize(act_name, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                cv2.rectangle(img, (mid_x, mid_y - text_size[1] - 2), (mid_x + text_size[0], mid_y + 2), (255,255,255), -1)
                
                cv2.putText(img, act_name, (mid_x, mid_y), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    # 5. Save
    out_path = os.path.join(output_dir, os.path.basename(item['file_name']))
    cv2.imwrite(out_path, img)

print("Visualizations updated with text labels!")

Visualizations updated with text labels!
